In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

# ------------------------------------------------------------
# INTRODUCTION
# ------------------------------------------------------------

intro_html = HTML("""
<div style="font-size:14px; line-height:1.55; width:980px; padding:8px 12px; margin-bottom:10px;">
<b>Interactive Notch Filter Explorer</b><br><br>
This notebook illustrates the frequency response of a second-order notch filter described by
the pole natural frequency <b>ω<sub>p</sub></b>, the zero frequency <b>ω<sub>z</sub></b>,
and the quality factor <b>Q</b>.
By varying these parameters, observe how the relative positions of ω<sub>p</sub> and
ω<sub>z</sub> determine the type of notch filter:
<b>low-pass notch</b> for ω<sub>z</sub> &gt; ω<sub>p</sub>,
<b>high-pass notch</b> for ω<sub>z</sub> &lt; ω<sub>p</sub>, and
<b>symmetric notch</b> for ω<sub>z</sub> = ω<sub>p</sub>.
The zero of the magnitude response always occurs at ω = ω<sub>z</sub>, while the quality
factor Q controls the sharpness of the response around the pole frequency.
</div>
""")

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

wp_title = HTML(value="<b>Pole Frequency ω<sub>p</sub> (rad/s)</b>")
wp_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='280px'))

wz_title = HTML(value="<b>Zero Frequency ω<sub>z</sub> (rad/s)</b>")
wz_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=3.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='280px'))

q_title = HTML(value="<b>Quality Factor Q</b>")
q_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='280px'))

wp_box = VBox([wp_title, wp_slider], layout=Layout(width='300px'))
wz_box = VBox([wz_title, wz_slider], layout=Layout(width='300px'))
q_box = VBox([q_title, q_slider], layout=Layout(width='300px'))

control_row = HBox([wp_box, wz_box, q_box], layout=Layout(width='940px', justify_content='space-between', align_items='flex-start'))

# ------------------------------------------------------------
# CALCULATED PARAMETERS
# ------------------------------------------------------------

parameter_title = HTML(value="<b>Current Filter Characteristics</b>")

type_output = HTML(layout=Layout(width='900px'))
gain_output = HTMLMath(layout=Layout(width='900px'))
frequency_output = HTMLMath(layout=Layout(width='900px'))

parameter_box = VBox([parameter_title, type_output, gain_output, frequency_output], layout=Layout(width='920px', margin='8px 0 8px 0'))

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_notch_filter(wp, wz, Q):

    A = 1.0

    # --------------------------------------------------------
    # FILTER CLASSIFICATION
    # --------------------------------------------------------

    if np.isclose(wz, wp, rtol=0.0, atol=1e-12):

        filter_name = "Symmetric notch filter"

    elif wz > wp:

        filter_name = "Low-pass notch filter"

    else:

        filter_name = "High-pass notch filter"

    type_output.value = "<div style='font-size:15px;'><b>Filter Type:</b> " + filter_name + "</div>"

    # --------------------------------------------------------
    # IMPORTANT GAIN VALUES
    # --------------------------------------------------------

    G0 = A * wz**2 / wp**2
    Ginf = A

    gain_output.value = rf"$$G(0)=A\frac{{\omega_z^2}}{{\omega_p^2}}={G0:.4f}\qquad\qquad G(\infty)=A={Ginf:.4f}$$"

    frequency_output.value = rf"$$\omega_p={wp:.2f}\ \mathrm{{rad/s}}\qquad\qquad\omega_z={wz:.2f}\ \mathrm{{rad/s}}\qquad\qquad Q={Q:.2f}$$"

    # --------------------------------------------------------
    # FREQUENCY RANGE
    # --------------------------------------------------------

    wmin = min(wp, wz) / 10.0
    wmax = max(wp, wz) * 10.0

    omega = np.logspace(np.log10(wmin), np.log10(wmax), 2000)

    # --------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------

    jw = 1j * omega

    H = A * (jw**2 + wz**2) / (jw**2 + (wp/Q)*jw + wp**2)

    magnitude = np.abs(H)

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(9.4, 4.2))

    ax.semilogx(omega, magnitude, 'r-', linewidth=2.0, label=r'$G(\omega)=|H(j\omega)|$')

    # --------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------

    ax.axvline(wp, color='gray', linestyle='--', linewidth=1.2, label=r'$\omega_p$')
    ax.axvline(wz, color='black', linestyle=':', linewidth=1.4, label=r'$\omega_z$')

    ax.axhline(A, color='gray', linestyle='--', linewidth=1.0)
    ax.axhline(G0, color='gray', linestyle=':', linewidth=1.0)

    # --------------------------------------------------------
    # ZERO LOCATION
    # --------------------------------------------------------

    ax.plot(wz, 0.0, 'o', markersize=6)

    # --------------------------------------------------------
    # AXES
    # --------------------------------------------------------

    ymax = max(1.15 * np.nanmax(magnitude), 1.25 * A, 1.25 * G0)

    ax.set_title(filter_name)
    ax.set_xlabel(r'Angular Frequency $\omega$ (rad/s)')
    ax.set_ylabel(r'Gain $G(\omega)$')
    ax.set_xlim(wmin, wmax)
    ax.set_ylim(0.0, ymax)
    ax.grid(True, which='both', linestyle=':', alpha=0.7)

    # Legend outside the graph, on the right
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

    fig.subplots_adjust(left=0.10, right=0.77, bottom=0.16, top=0.88)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_notch_filter, wp=wp_slider, wz=wz_slider, Q=q_slider)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

# ------------------------------------------------------------
# REMOVE OUTPUT SCROLL BARS
# ------------------------------------------------------------

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(intro_html)
display(control_row)
display(parameter_box)
display(plot_output)